# Multi-Factor Strategy

Multi-factor strategies combine signals such as value, momentum, and quality into one ranking system.

Abbreviations used in this notebook:

- **MOM**: Momentum.
- **ROIC**: Return on Invested Capital.
- **P/E**: Price to Earnings.
- **EV/EBITDA**: Enterprise Value divided by EBITDA.
- **EBITDA**: Earnings Before Interest, Taxes, Depreciation, and Amortization.
- **EV**: Enterprise Value, the value of the operating business before subtracting net debt.
- **IR**: Information Ratio.

## 1. Intuition

Different factors work in different environments. Combining factors can create a more balanced process than relying on one signal alone.

## 2. Mathematics

**Composite score:**

$$
Score = w_v Value + w_m Momentum + w_q Quality
$$

Where:

- $Score$ = composite ranking score
- $Value$ = value factor score or valuation output, depending on context
- $Momentum$ = momentum factor score
- $Quality$ = quality factor score
- $w_v$ = value factor weight
- $w_m$ = momentum factor weight
- $w_q$ = quality factor weight

**Where weights sum to one:**

$$
w_v + w_m + w_q = 1
$$

Where:

- $w_v$ = value factor weight
- $w_m$ = momentum factor weight
- $w_q$ = quality factor weight

## 3. Implementation

We combine value, momentum, and quality scores, then select the top stocks.

In [ ]:
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "GUIDELINES.md").exists())
helper_path = project_root / "05_strategies" / "strategy_utils.py"
spec = importlib.util.spec_from_file_location("strategy_utils", helper_path)
strategy_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(strategy_utils)

plt.style.use("seaborn-v0_8-whitegrid")
prices, returns, fundamentals = strategy_utils.generate_strategy_universe()
benchmark = returns.mean(axis=1)

latest_momentum = prices.pct_change(126).iloc[-1]
rank = fundamentals.copy().set_index("ticker")
rank["value_score"] = strategy_utils.zscore(rank["pe"], False) + strategy_utils.zscore(rank["ev_ebitda"], False)
rank["quality_score"] = strategy_utils.zscore(rank["roic"], True) + strategy_utils.zscore(rank["debt_to_ebitda"], False)
rank["momentum_score"] = strategy_utils.zscore(latest_momentum, True)
rank["multi_factor_score"] = 0.35 * rank["value_score"] + 0.35 * rank["quality_score"] + 0.30 * rank["momentum_score"]
rank = rank.sort_values("multi_factor_score", ascending=False)
selected = rank.head(8).index.tolist()
multi_returns = strategy_utils.equal_weight_return(returns, selected)
rank.head(10)

In [ ]:
component_returns = pd.DataFrame({
    "multi_factor": multi_returns,
    "benchmark": benchmark,
})
summary = pd.DataFrame({
    "multi_factor": strategy_utils.performance_summary(multi_returns, benchmark),
    "benchmark": strategy_utils.performance_summary(benchmark),
})
summary.round(4)

## 4. Visualization

A multi-factor process should show both the final score and the component scores.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
rank.head(10)[["value_score", "quality_score", "momentum_score"]].plot(kind="bar", ax=axes[0])
axes[0].set_title("Top Stocks by Component Scores")
axes[0].tick_params(axis="x", rotation=35)

(1 + component_returns).cumprod().plot(ax=axes[1], color=["#2f6f8f", "#9a6b2f"])
axes[1].set_title("Multi-Factor Strategy vs Benchmark")
axes[1].set_ylabel("Growth of 1")
plt.tight_layout(); plt.show()

## 5. Application

Multi-factor models are useful because they diversify signals. They also require governance: factor definitions, weights, rebalancing frequency, and risk controls must be explicit.

In [ ]:
rank.loc[selected, ["value_score", "quality_score", "momentum_score", "multi_factor_score", "pe", "roic"]].round(3)

## 6. Reflection

- Multi-factor strategies reduce dependence on one signal.
- Factor weights are assumptions and should be tested.
- Component conflicts are normal.
- Simpler models are often easier to maintain.

Questions to answer after running the notebook:

1. Which factor dominates the selected names?
2. Did the composite portfolio outperform?
3. What happens if momentum receives a higher weight?
4. How would you control sector concentration in a real model?